# 🧪 A/B 测试马拉松：真实场景实战 (The Real-World Scenarios)

> **目标**: 超越基础 CTR。 搞定漏斗 (Funnels)、留存 (Retention) 和样本量 (Sample Size) 计算。
> **状态**: 准备就绪，开始练习。

---

## 📉 Case 3: 漏斗转化分析 (Funnel Analysis)
**难度**: ⭐⭐⭐⭐ | **核心考点**: 卡方检验 (Chi-Square Test)

### 3.1 背景 (Context)
- **场景**: 我们简化了结账流程。每一步的转化率是否都有提升？
- **数据**: `df_funnel` 包含 `user_id`, `group`, 和 `max_step` (用户到达的最远步骤)。
- **步骤**: `View` (浏览) -> `Cart` (加购) -> `Pay` (支付) -> `Success` (成功)

### 3.2 任务 (Tasks)
1.  **聚合 (Aggregation)**: 计算每一步的用户数。*提示：到达 'Success' 的用户也算作 'View', 'Cart', 'Pay'。*
2.  **指标 (Metrics)**: 计算步骤间的转化率 (例如：加购 -> 支付率)。
3.  **检验 (Testing)**: 使用 `scipy.stats.chi2_contingency` 检验两组的漏斗分布是否存在显著差异。

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# --- 0. Case 3 造数据 ---
np.random.seed(99)
n = 3000
groups_c3 = np.random.choice(['Control', 'Test'], size=n)
steps = ['View', 'Cart', 'Pay', 'Success']
probs_control = [0.4, 0.3, 0.2, 0.1]
probs_test =    [0.35, 0.3, 0.2, 0.15] 
max_steps = [np.random.choice(steps, p=probs_control if g == 'Control' else probs_test) for g in groups_c3]
df_funnel = pd.DataFrame({'user_id': range(n), 'group': groups_c3, 'max_step': max_steps})
print("Case 3 数据准备就绪。")
# -------------------------------
print(df_funnel)

Case 3 数据准备就绪。
      user_id    group max_step
0           0     Test  Success
1           1     Test     Cart
2           2     Test     View
3           3  Control     View
4           4     Test      Pay
...       ...      ...      ...
2995     2995  Control     Cart
2996     2996     Test      Pay
2997     2997     Test     View
2998     2998  Control     Cart
2999     2999     Test     View

[3000 rows x 3 columns]


In [2]:
# --- 你的 Case 3 代码 ---
# 1. 定义逻辑顺序 (List)
step_order = ['View', 'Cart', 'Pay', 'Success']
df_funnel['max_step'] = pd.Categorical(df_funnel['max_step'], categories=step_order, ordered=True)
df_funnel = df_funnel.sort_values('max_step')


In [3]:
df_user_cnt = df_funnel.groupby(['group','max_step'])['user_id'].count().unstack()
df_user_cnt

/var/folders/35/q6rh83x91bzgf3gcsb_13f_80000gn/T/ipykernel_17539/1468758260.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_user_cnt = df_funnel.groupby(['group','max_step'])['user_id'].count().unstack()


max_step,View,Cart,Pay,Success
group,,,,
Control,573,464,281,139
Test,533,460,327,223


In [4]:
df_user_cnt = df_user_cnt.reset_index()
df_user_cnt['cart/view'] = df_user_cnt['Cart']/df_user_cnt['View']
df_user_cnt['pay/cart'] = df_user_cnt['Pay']/df_user_cnt['Cart']
df_user_cnt['success/pay'] = df_user_cnt['Success']/df_user_cnt['Pay']
df_user_cnt

max_step,group,View,Cart,Pay,Success,cart/view,pay/cart,success/pay
0,Control,573,464,281,139,0.809773,0.605603,0.494662
1,Test,533,460,327,223,0.863039,0.710870,0.681957


In [5]:
total_users = df_user_cnt['View'].sum()
df_user_cnt['group%'] = df_user_cnt['View']/total_users
df_user_cnt['final_funnel'] = df_user_cnt['Success']/df_user_cnt['View']
df_user_cnt

max_step,group,View,Cart,Pay,Success,cart/view,pay/cart,success/pay,group%,final_funnel
0,Control,573,464,281,139,0.809773,0.605603,0.494662,0.518083,0.242583
1,Test,533,460,327,223,0.863039,0.710870,0.681957,0.481917,0.418386


In [6]:
import numpy as np

# --- 1. 准备数据 ---
# 卡方检验必须用【具体人数(Counts)】，不能用比率！
# 我们需要构建一个 2x2 的观察矩阵：[[成功人数, 失败人数], [成功人数, 失败人数]]

# 提取 Control 组的数值
c_data = df_user_cnt[df_user_cnt['group'] == 'Control']
c_success = c_data['Success'].values[0]
c_fail = c_data['View'].values[0] - c_success  # 总人数(View) - 成功人数 = 失败人数

# 提取 Test 组的数值
t_data = df_user_cnt[df_user_cnt['group'] == 'Test']
t_success = t_data['Success'].values[0]
t_fail = t_data['View'].values[0] - t_success

# 构建列联表 (Observed Matrix)
obs = np.array([
    [c_success, c_fail],  # Control组
    [t_success, t_fail]   # Test组
])

print("构建的列联表 (Observed Table):")
print(obs)

# --- 2. 运行卡方检验 ---
# 注意：这里必须接收 4 个返回值，不管你想不想要后两个
chi2_stat, p_val, dof, expected = stats.chi2_contingency(obs)

print(f"\n卡方统计量 (Chi2): {chi2_stat:.4f}")
print(f"P值 (P-value): {p_val:.4e}")  # 科学计数法，方便看极小值

# 判断结果
alpha = 0.05
if p_val < alpha:
    print("结论：拒绝原假设，两组转化率有显著差异 (Significant)")
else:
    print("结论：无法拒绝原假设，无显著差异")

构建的列联表 (Observed Table):
[[139 434]
 [223 310]]

卡方统计量 (Chi2): 37.9681
P值 (P-value): 7.1912e-10
结论：拒绝原假设，两组转化率有显著差异 (Significant)


## 📅 Case 4: 留存率分析 (Cohort Analysis)
**难度**: ⭐⭐⭐ | **核心考点**: 日期处理, 透视表 (Pivot Table), T-Test

### 4.1 背景 (Context)
- **场景**: 新的新手引导流程。它是否提升了次日留存 (Day-1) 和 7日留存 (Day-7)？
- **数据**: `df_login_logs` (user_id, login_date), `df_signup` (user_id, signup_date, group)。

### 4.2 任务 (Tasks)
1.  **关联 (Merge)**: 将日志表与注册表关联。
2.  **天数差 (Day Diff)**: 计算 `days_since_signup` = `login_date` - `signup_date`。
3.  **留存标记 (Retention Flag)**: 创建 `is_retained_d1`, `is_retained_d7` 标记。
4.  **检验 (Test)**: 对 7日留存率进行 T-Test。

In [7]:
# --- 0. Case 4 造数据 ---
np.random.seed(42)
n_users = 2000
groups = np.random.choice(['Control', 'Test'], size=n_users)
signup_dates = pd.to_datetime('2024-01-01') + pd.to_timedelta(np.random.randint(0, 10, n_users), unit='D')
df_signup = pd.DataFrame({'user_id': range(n_users), 'group': groups, 'signup_date': signup_dates})

# 生成登录日志
logs = []
for uid, s_date, grp in zip(df_signup['user_id'], df_signup['signup_date'], df_signup['group']):
    # 基础留存概率
    p = 0.4 if grp == 'Control' else 0.45
    for day in [1, 3, 7, 14, 30]:
        if np.random.rand() < (p / (day**0.5)): # 随时间衰减
            logs.append({'user_id': uid, 'login_date': s_date + pd.Timedelta(days=day)})
df_logs = pd.DataFrame(logs)
print("Case 4 数据准备就绪。")
# -------------------------------

Case 4 数据准备就绪。


In [8]:
# --- 你的 Case 4 代码 ---
df_logs['user_id'] = df_logs['user_id'].astype(str)
df_signup['user_id'] = df_signup['user_id'].astype(str)
print(df_logs.describe(include='all'))
print(df_logs.head())
print(df_signup.describe(include='all'))
print(df_signup.head())

       user_id                     login_date
count     2023                           2023
unique    1373                            NaN
top        242                            NaN
freq         4                            NaN
mean       NaN  2024-01-11 15:02:34.819574784
min        NaN            2024-01-02 00:00:00
25%        NaN            2024-01-06 00:00:00
50%        NaN            2024-01-09 00:00:00
75%        NaN            2024-01-15 00:00:00
max        NaN            2024-02-09 00:00:00
  user_id login_date
0       0 2024-01-11
1       1 2024-01-04
2       1 2024-01-10
3       3 2024-01-06
4       3 2024-01-12
       user_id    group                    signup_date
count     2000     2000                           2000
unique    2000        2                            NaN
top          0  Control                            NaN
freq         1     1016                            NaN
mean       NaN      NaN  2024-01-05 10:30:43.199999744
min        NaN      NaN            202

In [9]:
# 1.  **关联 (Merge)**: 将日志表与注册表关联。
df_user_logs = df_logs.groupby('user_id')['login_date'].max().reset_index()
df_user_logs.describe(include='all')

df_signup_merge = df_signup.merge(df_user_logs,how='left',on='user_id')

In [10]:
# 2.  **天数差 (Day Diff)**: 计算 `days_since_signup` = `login_date` - `signup_date`。
df_signup_merge['days_since_signup'] = (df_signup_merge['login_date'] - df_signup_merge['signup_date']).dt.days
df_signup_merge

,user_id,group,signup_date,login_date,days_since_signup
0,0,Control,2024-01-08,2024-01-11,3.0
1,1,Test,2024-01-03,2024-01-10,7.0
2,2,Control,2024-01-08,NaT,NaN
3,3,Control,2024-01-05,2024-01-12,7.0
4,4,Control,2024-01-01,2024-01-02,1.0
...,...,...,...,...,...
1995,1995,Control,2024-01-06,NaT,NaN
1996,1996,Test,2024-01-04,2024-01-11,7.0
1997,1997,Control,2024-01-07,2024-01-21,14.0
1998,1998,Control,2024-01-08,NaT,NaN


In [11]:
# 3.  **留存标记 (Retention Flag)**: 创建 `is_retained_d1`, `is_retained_d7` 标记。
df_signup_merge['is_retained_d1'] = np.where(df_signup_merge['days_since_signup']>=1,1,0)
df_signup_merge['is_retained_d7'] = np.where(df_signup_merge['days_since_signup']>=7,1,0)
df_signup_merge

,user_id,group,signup_date,login_date,days_since_signup,is_retained_d1,is_retained_d7
0,0,Control,2024-01-08,2024-01-11,3.0,1,0
1,1,Test,2024-01-03,2024-01-10,7.0,1,1
2,2,Control,2024-01-08,NaT,NaN,0,0
3,3,Control,2024-01-05,2024-01-12,7.0,1,1
4,4,Control,2024-01-01,2024-01-02,1.0,1,0
...,...,...,...,...,...,...,...
1995,1995,Control,2024-01-06,NaT,NaN,0,0
1996,1996,Test,2024-01-04,2024-01-11,7.0,1,1
1997,1997,Control,2024-01-07,2024-01-21,14.0,1,1
1998,1998,Control,2024-01-08,NaT,NaN,0,0


In [12]:
# 4.  **检验 (Test)**: 对 7日留存率进行 T-Test。
import scipy.stats as stats
control_group = df_signup_merge[df_signup_merge['group']=='Control']
test_group = df_signup_merge[df_signup_merge['group']=='Test']
# control_group
result = stats.ttest_ind(control_group['is_retained_d7'],test_group['is_retained_d7'])

print(f"P值 (P-value): {round(result[1],4)}")  

# 判断结果
alpha = 0.05
if result[1] < alpha:
    print("结论：有显著差异")
else:
    print("结论：无显著差异")


P值 (P-value): 0.091
结论：无显著差异


## 🔍 Case 5: 样本量估算 (Power Analysis)
**难度**: ⭐⭐⭐⭐⭐ | **核心考点**: 功效 (Power), 显著性水平 (Alpha), MDE (最小可检测效应)

### 5.1 背景 (Context)
- **场景**: 在开始实验前，产品经理问：“我们需要多少用户才能检测到转化率提升 1%？” (基线转化率为 10%)。
- **工具**: `statsmodels.stats.power.NormalIndPower`

### 5.2 任务 (Tasks)
1.  **定义参数**: Alpha=0.05, Power=0.8, 效应量 (Standardized Effect Size)。
2.  **计算**: 使用 `solve_power` 计算 `nobs1` (也就是样本量)。
3.  **可视化**: 绘制 “功效曲线 (Power Curve)” (样本量 vs. 可检测效应)。

In [14]:
from statsmodels.stats.power import NormalIndPower
# --- 你的 Case 5 代码 ---
import numpy as np
import statsmodels.stats.api as sms


# 1. Effect Size (效应量): 这里用的是 Cohen's h
# p1 = Baseline (当前水平): 比如现在页面转化率是 10%
p1 = 0.10
# p2 = Target (目标水平): 我们希望新版本能达到 12% (即提升 2pp)
p2 = 0.11

effect_size = sms.proportion_effectsize(p1, p2)     

# 2. 算样本量
required_n = sms.NormalIndPower().solve_power(
    effect_size=effect_size, 
    power=0.8, 
    alpha=0.05, 
    ratio=1  # 实验组和对照组 1:1 分配
)

print(f"我们需要每组至少 {int(np.ceil(required_n))} 个样本。")


我们需要每组至少 14745 个样本。
